# 🦆 Session 2: DuckDB Fundamental

**Durasi:** 90 menit  
**Dataset:** RUP Paket Penyedia 2025

## 🎯 Tujuan Pembelajaran
Setelah sesi ini, Anda dapat:
1. Menjalankan SQL query dengan DuckDB
2. Melakukan SELECT, WHERE, ORDER BY
3. Menggunakan aggregate functions (SUM, COUNT, AVG)
4. Melakukan JOIN antar tabel
5. Export hasil query

## 1️⃣ Setup DuckDB

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

# Inisialisasi DuckDB (in-memory)
conn = duckdb.connect(':memory:')
print(f"✅ DuckDB version: {duckdb.__version__}")

✅ DuckDB version: 1.4.1


In [2]:
# Load data dan register ke DuckDB
data_path = Path('../../../../datasets/StudentPerformanceFactors.csv')
df = pd.read_csv(data_path)

# Register DataFrame sebagai tabel
conn.register('rup', df)

print(f"✅ Data registered: {len(df):,} rows")

✅ Data registered: 6,607 rows


## 2️⃣ Query Dasar: SELECT & WHERE

In [3]:


# Daftarkan DataFrame sebagai tabel 'students'
conn.register('students', df)



In [4]:
# Query: 5 baris pertama (kolom relevan)
query = """
SELECT 
    Gender,
    School_Type,
    Hours_Studied,
    Exam_Score
FROM students
LIMIT 5
"""

result = conn.execute(query).df()
print(result)

   Gender School_Type  Hours_Studied  Exam_Score
0    Male      Public             23          67
1  Female      Public             19          61
2    Male      Public             24          74
3    Male      Public             29          71
4  Female      Public             19          70


In [5]:
# Query 3: Filter dengan kondisi AND
query = """
SELECT COUNT(*) AS jumlah_paket,
       SUM(pagu) / 1e9 AS total_pagu_miliar
FROM rup
WHERE metode_pengadaan = 'Tender'
  AND pagu > 1000000000
"""

conn.execute(query).df()

BinderException: Binder Error: Referenced column "metode_pengadaan" not found in FROM clause!
Candidate bindings: "Gender", "Peer_Influence", "Parental_Education_Level", "Sleep_Hours", "Tutoring_Sessions"

LINE 5: WHERE metode_pengadaan = 'Tender'
              ^

## 3️⃣ Aggregate Functions

In [19]:
# Statistik pagu per metode pengadaan
query = """
SELECT metode_pengadaan,
       COUNT(*) AS jumlah_paket,
       SUM(pagu) / 1e9 AS total_pagu_miliar,
       AVG(pagu) / 1e6 AS rata_pagu_juta,
       MIN(pagu) AS pagu_min,
       MAX(pagu) / 1e9 AS pagu_max_miliar
FROM rup
GROUP BY metode_pengadaan
ORDER BY total_pagu_miliar DESC
"""

conn.execute(query).df()

,metode_pengadaan,jumlah_paket,total_pagu_miliar,rata_pagu_juta,pagu_min,pagu_max_miliar
0,Pengadaan Langsung,4515,706.994611,156.587954,22000,23.634000
1,E-Purchasing,11245,644.444465,57.309423,1,55.565778
2,Tender,77,569.147598,7391.527241,18380000,37.000000
3,Dikecualikan,497,148.135926,298.060213,50000,74.802425
4,Seleksi,44,32.411774,736.631236,172500000,1.800000
5,Penunjukan Langsung,48,7.201475,150.030724,1000000,1.500000
6,Tender Cepat,3,3.328240,1109.413333,613200000,1.948000
7,Kontes,1,0.001984,1.984000,1984000,0.001984


In [7]:
# Top 10 Satker
query = """
SELECT nama_satker,
       COUNT(*) AS jumlah_paket,
       SUM(pagu) / 1e9 AS total_pagu_miliar,
       ROUND(AVG(pagu) / 1e6, 2) AS rata_pagu_juta
FROM rup
GROUP BY nama_satker
ORDER BY total_pagu_miliar DESC
LIMIT 10
"""

conn.execute(query).df()

,nama_satker,jumlah_paket,total_pagu_miliar,rata_pagu_juta
0,DINAS PEKERJAAN UMUM DAN PENATAAN RUANG PROVIN...,2428,841.492369,346.58
1,RUMAH SAKIT UMUM DAERAH DOKTER SOEDARSO,231,331.063820,1433.18
2,DINAS PERUMAHAN RAKYAT DAN KAWASAN PERMUKIMAN,1711,282.709740,165.23
3,DINAS PENDIDIKAN DAN KEBUDAYAAN PROVINSI KALIM...,688,203.875270,296.33
4,SEKRETARIAT DEWAN PERWAKILAN RAKYAT DAERAH PRO...,367,71.445026,194.67
5,SEKRETARIAT DAERAH PROVINSI KALBAR,283,48.242939,170.47
6,RUMAH SAKIT JIWA PROVINSI KALIMANTAN BARAT,104,43.546135,418.71
7,DINAS TANAMAN PANGAN DAN HORTIKULTURA PROVINSI...,288,23.941928,83.13
8,DINAS KESEHATAN PROVINSI KALIMANTAN BARAT,251,17.933055,71.45
9,BADAN KEUANGAN DAN ASET DAERAH PROVINSI KALIMA...,567,15.665977,27.63


## 4️⃣ Filtering dengan HAVING

In [8]:
# Satker dengan total pagu > 10 Miliar
query = """
SELECT nama_satker,
       COUNT(*) AS jumlah_paket,
       SUM(pagu) / 1e9 AS total_pagu_miliar
FROM rup
GROUP BY nama_satker
HAVING SUM(pagu) > 10000000000
ORDER BY total_pagu_miliar DESC
"""

conn.execute(query).df()

,nama_satker,jumlah_paket,total_pagu_miliar
0,DINAS PEKERJAAN UMUM DAN PENATAAN RUANG PROVIN...,2428,841.492369
1,RUMAH SAKIT UMUM DAERAH DOKTER SOEDARSO,231,331.063820
2,DINAS PERUMAHAN RAKYAT DAN KAWASAN PERMUKIMAN,1711,282.709740
3,DINAS PENDIDIKAN DAN KEBUDAYAAN PROVINSI KALIM...,688,203.875270
4,SEKRETARIAT DEWAN PERWAKILAN RAKYAT DAERAH PRO...,367,71.445026
5,SEKRETARIAT DAERAH PROVINSI KALBAR,283,48.242939
6,RUMAH SAKIT JIWA PROVINSI KALIMANTAN BARAT,104,43.546135
7,DINAS TANAMAN PANGAN DAN HORTIKULTURA PROVINSI...,288,23.941928
8,DINAS KESEHATAN PROVINSI KALIMANTAN BARAT,251,17.933055
9,BADAN KEUANGAN DAN ASET DAERAH PROVINSI KALIMA...,567,15.665977


## 5️⃣ CASE Statement

In [9]:
# Kategorisasi paket berdasarkan nilai pagu
query = """
SELECT 
    CASE 
        WHEN pagu < 10000000 THEN 'Kecil (< 10 Juta)'
        WHEN pagu < 100000000 THEN 'Menengah (10-100 Juta)'
        WHEN pagu < 1000000000 THEN 'Besar (100 Juta - 1 M)'
        ELSE 'Sangat Besar (> 1 M)'
    END AS kategori_pagu,
    COUNT(*) AS jumlah_paket,
    SUM(pagu) / 1e9 AS total_pagu_miliar
FROM rup
GROUP BY kategori_pagu
ORDER BY total_pagu_miliar DESC
"""

conn.execute(query).df()

,kategori_pagu,jumlah_paket,total_pagu_miliar
0,Sangat Besar (> 1 M),205,1181.562461
1,Besar (100 Juta - 1 M),3834,768.229731
2,Menengah (10-100 Juta),3876,141.210554
3,Kecil (< 10 Juta),8515,20.663325


## 6️⃣ String Functions

In [10]:
# Cari paket yang mengandung kata "Belanja"
query = """
SELECT nama_paket,
       pagu / 1e6 AS pagu_juta,
       metode_pengadaan
FROM rup
WHERE LOWER(nama_paket) LIKE '%belanja%'
LIMIT 10
"""

conn.execute(query).df()

,nama_paket,pagu_juta,metode_pengadaan
0,Belanja Bahan Makanan dan Minuman Pasien,7700.0000,Tender
1,Belanja Alat/Bahan untuk Kegiatan Kantor- Kert...,2.1760,E-Purchasing
2,Belanja Alat/Bahan untuk Kegiatan Kantor-Alat ...,5.3398,E-Purchasing
3,Belanja Alat/Bahan untuk Kegiatan Kantor- Baha...,2.8400,E-Purchasing
4,Belanja Alat/Bahan untuk Kegiatan Kantor-Bahan...,8.6454,E-Purchasing
5,Belanja Makanan dan Minuman Rapat 5.02.01.1.02...,5.4400,E-Purchasing
6,Belanja Alat/Bahan untuk Kegiatan Kantor- Baha...,1.4500,E-Purchasing
7,Belanja Alat/Bahan untuk Kegiatan Kantor- Kert...,2.1750,E-Purchasing
8,Belanja Alat/Bahan untuk Kegiatan Kantor-Bahan...,5.5740,E-Purchasing
9,Belanja Makanan dan Minuman Rapat 5.02.01.1.02...,4.3520,E-Purchasing


## 7️⃣ Export Results

In [11]:
# Export ke CSV
query = """
COPY (
    SELECT metode_pengadaan,
           COUNT(*) AS jumlah_paket,
           SUM(pagu) / 1e9 AS total_pagu_miliar
    FROM rup
    GROUP BY metode_pengadaan
) TO 'summary_metode.csv' (HEADER, DELIMITER ',')
"""

conn.execute(query)
print("✅ Data exported to summary_metode.csv")

✅ Data exported to summary_metode.csv


## 📊 Query Kompleks: Multi-Level Aggregation

In [12]:
# Analisis metode dan jenis pengadaan
query = """
SELECT metode_pengadaan,
       jenis_pengadaan,
       COUNT(*) AS jumlah_paket,
       ROUND(SUM(pagu) / 1e9, 2) AS total_pagu_miliar,
       ROUND(AVG(pagu) / 1e6, 2) AS rata_pagu_juta
FROM rup
WHERE jenis_pengadaan IS NOT NULL
GROUP BY metode_pengadaan, jenis_pengadaan
HAVING COUNT(*) > 10
ORDER BY metode_pengadaan, total_pagu_miliar DESC
"""

result = conn.execute(query).df()
result.head(15)

,metode_pengadaan,jenis_pengadaan,jumlah_paket,total_pagu_miliar,rata_pagu_juta
0,Dikecualikan,Barang,234,125.35,535.67
1,Dikecualikan,Jasa Lainnya,263,22.79,86.65
2,E-Purchasing,Barang,9983,507.55,50.84
3,E-Purchasing,Jasa Lainnya,1183,129.19,109.21
4,E-Purchasing,Pekerjaan Konstruksi,46,6.73,146.32
5,E-Purchasing,Jasa Konsultansi,31,0.93,30.03
6,Pengadaan Langsung,Pekerjaan Konstruksi,2780,565.03,203.25
7,Pengadaan Langsung,Jasa Konsultansi,1237,76.13,61.55
8,Pengadaan Langsung,Jasa Lainnya,201,43.42,216.04
9,Pengadaan Langsung,Barang,297,22.41,75.45


## 🎯 Latihan Mandiri

1. Hitung jumlah paket untuk setiap jenis pengadaan
2. Temukan 5 paket dengan nama terpanjang
3. Hitung persentase paket per metode pengadaan
4. Buat query untuk menemukan satker dengan rata-rata pagu terbesar (min 10 paket)

In [13]:
# Ruang untuk latihan

In [14]:
# Tutup koneksi
conn.close()
print("✅ Connection closed")

✅ Connection closed
